In [5]:
import pandas as pd
import requests
import json
import geopandas as gpd
import numpy as np
from shapely.geometry import Polygon 



In [7]:
demographics = json.loads(requests.get('https://files.jcrayb.com/files/ie300/majority.json').text)

In [11]:
chicago = gpd.read_file('osmnx/data/zipcodes.geojson').dropna(subset='zip')[['zip', 'geometry']]

In [12]:
centroids = {}

tracts = gpd.read_file('osmnx/data/tracts.geojson')



tract_demographics = {}

zips = chicago.zip.to_list()

zips = [zip for zip in zips if zip]

demographics = {zip: majority for zip, majority in demographics.items() if zip in zips}

for tract in tracts.iloc:
    centroid = tract.geometry.centroid

    for row in chicago.iloc:
        zip_geometry = row.geometry

        if zip_geometry.contains(centroid) and (row.zip in demographics):
            tract_demographics[tract.namelsad10] = demographics[row.zip]
            continue


In [15]:
json.dump(tract_demographics, open('data/tract_demographics.json', 'w'), indent=2)

In [76]:
query = 'hospital'

destination = json.load(open(f'computation_results/budget_paths/{query}_final_results.json', 'r'))

n = 1

## Includes no cameras

In [77]:


df = pd.DataFrame(index=list(tract_demographics.keys()), columns=['majority', 'shortest_path', 'shortest_restricted_path', 'relative_increase', 'absolute_increase', 'n_cameras'])



for tract, race in tract_demographics.items():
    #print(tract, race)
    sorted_results = sorted(destination[tract], key=lambda x: list(destination[tract][x].items())[0][1])
    
    first_n_res = sorted_results[:n]
    n_cameras = []

    unrestricted_time_to_reach = []
    restricted_time_to_reach = []

    for place in first_n_res:
        results = destination[tract][place]
        #print(results)

        budgets = list(results.keys())
        highest, lowest = (budgets[0], budgets[-1])
        #print(highest, lowest)

        n_cameras += [int(highest)]
        unrestricted_time_to_reach += [results[highest]]
        restricted_time_to_reach += [results[lowest]]
    
    mean_uttr = np.mean(unrestricted_time_to_reach)
    mean_rttr = np.mean(restricted_time_to_reach)

    df.loc[df.index == tract] = [race, mean_uttr, mean_rttr, (mean_rttr-mean_uttr)/mean_uttr, mean_rttr-mean_uttr, np.mean(n_cameras)]

df.to_csv(f'./analysis/{query}-{n}-closest.csv')

## Only compares with cameras

In [78]:
df = pd.DataFrame(index=list(tract_demographics.keys()), columns=['majority', 'shortest_path', 'shortest_restricted_path', 'relative_increase', 'absolute_increase', 'n_cameras'])

for tract, race in tract_demographics.items():
    print(tract, race)
    sorted_results = sorted(destination[tract], key=lambda x: list(destination[tract][x].items())[0][1])
    
    first_n_res = sorted_results[:n]
    n_cameras = []

    unrestricted_time_to_reach = []
    restricted_time_to_reach = []

    for place in first_n_res:
        results = destination[tract][place]
        

        budgets = list(results.keys())
        highest, lowest = (budgets[0], budgets[-1])
        if highest == lowest: continue
        #print(highest, lowest)
        print(results)
        n_cameras += [int(highest)]
        unrestricted_time_to_reach += [results[highest]]
        restricted_time_to_reach += [results[lowest]]
    
    mean_uttr = np.mean(unrestricted_time_to_reach)
    mean_rttr = np.mean(restricted_time_to_reach)

    df.loc[df.index == tract] = [race, mean_uttr, mean_rttr, (mean_rttr-mean_uttr)/mean_uttr, mean_rttr-mean_uttr, np.mean(n_cameras)]

df.to_csv(f'./analysis/{query}-{n}-closest-nonzero.csv')

Census Tract 8424 black
Census Tract 8403 white
{'1': 339.20000000000005, '0': 339.7}
Census Tract 8411 white
Census Tract 8412 white
Census Tract 8390 white
Census Tract 8382 black
Census Tract 6503.01 white
{'1': 206.4, '0': 215.8}
Census Tract 5305.03 black
Census Tract 7608.03 white
Census Tract 306.01 white
Census Tract 306.04 white
{'1': 70.5, '0': 251.0}
Census Tract 208.01 white
Census Tract 5401.02 black
Census Tract 8433 white
Census Tract 5401.01 black
Census Tract 4402.01 black
{'1': 250.3, '0': 282.0}
Census Tract 802.02 white
Census Tract 701.02 white
Census Tract 315.01 white
Census Tract 315.02 white
Census Tract 8349 black
Census Tract 8348 black
Census Tract 1605.02 white
Census Tract 1407.02 white
Census Tract 8420 white
Census Tract 1504.02 white
{'2': 141.7, '1': 164.1}
Census Tract 8344 black
Census Tract 402.01 white
Census Tract 402.02 white
Census Tract 207.02 white
Census Tract 208.02 white
{'1': 111.30000000000001, '0': 121.2}
Census Tract 203.01 white
Census

/home/chris/.local/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/chris/.local/lib/python3.11/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [112]:
total_results

{'black': {'Avg time to reach destination': 824.3639670591363,
  'Avg time to reach destination while avoiding cameras': 852.8182276620697,
  '% difference in time': 0.0354895450972817,
  'absolute difference in time': 28.45426060293337,
  'Avg n cameras': 1.2389157832191888},
 'white': {'Avg time to reach destination': 743.9999773094596,
  'Avg time to reach destination while avoiding cameras': 787.6389687723249,
  '% difference in time': 0.059355610938969974,
  'absolute difference in time': 43.63899146286524,
  'Avg n cameras': 2.052083502024882}}